# 03 Radius-Distribution Robustness

## Purpose

This notebook tests one robustness case for the calibration framework used to separate

\[
n=\text{source-fluid virion concentration}
\]

and

\[
q=\text{per-delivered-virion probability of successful infection initiation}.
\]

The current single-hit model identifies only

\[
\phi=nq.
\]

Notebook 02 reproduces the baseline case in which

\[
f_{R,\mathrm{cal}}(r)
=
f_{R,\mathrm{vivo}}(r).
\]

This notebook tests the more general case

\[
f_{R,\mathrm{cal}}(r)
\neq
f_{R,\mathrm{vivo}}(r),
\]

while retaining the shared assumptions

\[
q_{\mathrm{cal}}
=
q_{\mathrm{vivo}}
=
q
\]

and

\[
\delta_{\mathrm{cal}}(r)
=
\delta_{\mathrm{vivo}}(r)
=
\delta(r).
\]

The calibration radius distribution uses

\[
r_{\mathrm{median,cal}}=0.18\ \mu\mathrm{m},
\qquad
\mathrm{GSD}_{\mathrm{cal}}=1.64,
\]

whereas the in vivo radius distribution remains

\[
r_{\mathrm{median,vivo}}=0.139\ \mu\mathrm{m},
\qquad
\mathrm{GSD}_{\mathrm{vivo}}=1.64.
\]

The agreed calibration quantities remain

\[
n_{\mathrm{cal}}^{\mathrm{true}}
=
1.0\times10^{12}\ \mathrm{virions/mL},
\]

\[
\phi_{\mathrm{cal}}^{\mathrm{true}}
=
6.696700846319259\times10^{10},
\]

\[
\mu_{\mathrm{cal}}=350,
\qquad
K_{\mathrm{cal}}=200.
\]

The corresponding true value of \(q\) is derived as

\[
q^{\mathrm{true}}
=
\frac{\phi_{\mathrm{cal}}^{\mathrm{true}}}
{n_{\mathrm{cal}}^{\mathrm{true}}}.
\]

The in vivo simulation uses

\[
n_{\mathrm{vivo}}^{\mathrm{true}}
=
5.0\times10^{11}\ \mathrm{virions/mL},
\]

with

\[
\phi_{\mathrm{vivo}}^{\mathrm{true}}
=
n_{\mathrm{vivo}}^{\mathrm{true}}q^{\mathrm{true}}.
\]

The in vivo simulation uses

\[
\mu_k\in\{100,200,350,500,700\},
\]

with

\[
N_k=200
\]

replicate trials at each specified value.

## Robustness analysis sequence

1. Construct \(f_{R,\mathrm{cal}}(r)\) and \(f_{R,\mathrm{vivo}}(r)\) separately.
2. Generate calibration outcomes using \(f_{R,\mathrm{cal}}(r)\).
3. Fit the current one-parameter Stan model using the calibration radius distribution to estimate
   \[
   \phi_{\mathrm{cal}}.
   \]
4. Recover
   \[
   q^{(s)}
   =
   \frac{\phi_{\mathrm{cal}}^{(s)}}
   {n_{\mathrm{cal}}^{\mathrm{true}}}.
   \]
5. Generate in vivo outcomes using \(f_{R,\mathrm{vivo}}(r)\).
6. Fit the same Stan model using the in vivo radius distribution to estimate
   \[
   \phi_{\mathrm{vivo}}.
   \]
7. Recover
   \[
   n_{\mathrm{vivo}}^{(s)}
   =
   \frac{\phi_{\mathrm{vivo}}^{(s)}}
   {q^{(s)}}.
   \]
8. Compare the recovered \(q\) and \(n_{\mathrm{vivo}}\) with their known simulation true values.

The purpose is to test whether the calibration framework remains valid when the calibration and in vivo aerosol-radius distributions differ, provided that the correct distribution is used in each corresponding model fit.

In [ ]:
# Local output directories used only for Stan compilation and sampling files
from pathlib import Path

OUTPUT_DIR = (
    Path.cwd()
    / "outputs"
    / "03_Radius_Distribution_Robustness"
)

CALIBRATION_CMDSTAN_DIR = OUTPUT_DIR / "calibration_cmdstan_csv"
VIVO_CMDSTAN_DIR = OUTPUT_DIR / "in_vivo_cmdstan_csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# @title Cell 2 — Install CmdStanPy and import libraries
import sys
import subprocess
import time
import shutil

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "cmdstanpy",
        "arviz",
    ]
)

import numpy as np
import pandas as pd
import arviz as az
import cmdstanpy

from cmdstanpy import CmdStanModel

CMDSTAN_VERSION = "2.40.0"

cmdstan_dir = (
    Path("/root/.cmdstan")
    / f"cmdstan-{CMDSTAN_VERSION}"
)

if not cmdstan_dir.exists():
    cmdstanpy.install_cmdstan(
        version=CMDSTAN_VERSION,
        cores=2,
        overwrite=False,
    )

cmdstanpy.set_cmdstan_path(
    str(cmdstan_dir)
)

print(
    f"CmdStan path: "
    f"{cmdstanpy.cmdstan_path()}"
)

In [ ]:
# @title Cell 3 — Define the simulation truth and simulation design

# Calibration truth
n_cal_true = 1.0e12

phi_cal_true = (
    6.696700846319259e10
)

# q_true is derived from the calibration quantities.
q_true = (
    phi_cal_true
    / n_cal_true
)

# One fixed calibration setting
mu_cal = 350.0
K_cal = 200

# In vivo truth
n_vivo_true = 5.0e11

phi_vivo_true = (
    n_vivo_true
    * q_true
)

# In vivo specified values of the expected aerosol count
mu_k = np.array(
    [
        100.0,
        200.0,
        350.0,
        500.0,
        700.0,
    ],
    dtype=float,
)

# Number of replicate trials at each mu_k
N_k = np.full(
    shape=len(mu_k),
    fill_value=200,
    dtype=int,
)

# Stan parameter scale only
phi_reference = 1.0e11

# Reproducible random seeds
SEED_CALIBRATION = 20260924
SEED_VIVO = 20260925
SEED_HMC_CALIBRATION = 20260926
SEED_HMC_VIVO = 20260927
SEED_POSTERIOR_COMBINATION = 20260928

print(
    f"n_cal_true   = "
    f"{n_cal_true:.6e} virions/mL"
)

print(
    f"phi_cal_true = "
    f"{phi_cal_true:.6e}"
)

print(
    f"q_true       = "
    f"{q_true:.8f}"
)

print(
    f"n_vivo_true  = "
    f"{n_vivo_true:.6e} virions/mL"
)

print(
    f"phi_vivo_true = "
    f"{phi_vivo_true:.6e}"
)

print(
    f"mu_cal = {mu_cal:.1f}, "
    f"K_cal = {K_cal}"
)

print(
    "mu_k =",
    mu_k.tolist(),
)

print(
    "N_k =",
    N_k.tolist(),
)

In [ ]:
# @title Cell 4 — Construct separate calibration and in vivo radius distributions

tau = 240.0
kappa_vent = 0.00833

r_min = 0.15
r_max = 2.50

# Calibration radius distribution
r_median_cal = 0.18
GSD_cal = 1.64

# In vivo radius distribution
r_median_vivo = 0.139
GSD_vivo = 1.64

# Shared baseline delivery probability
delta = 0.5

# Common quadrature grid
N_R = 1500

r_grid = np.linspace(
    r_min,
    r_max,
    N_R,
)


def f_emit(
    r,
    r_median,
    GSD,
):
    r = np.asarray(
        r,
        dtype=float,
    )

    mu_emit = np.log(
        r_median
    )

    sigma_emit = np.log(
        GSD
    )

    return (
        1.0
        / (
            r
            * sigma_emit
            * np.sqrt(
                2.0
                * np.pi
            )
        )
        * np.exp(
            -(
                (
                    np.log(r)
                    - mu_emit
                )
                ** 2
            )
            / (
                2.0
                * sigma_emit**2
            )
        )
    )


def kappa_dep(r):
    r = np.asarray(
        r,
        dtype=float,
    )

    return np.select(
        [
            (
                (r >= 0.15)
                & (r < 0.25)
            ),
            (
                (r >= 0.25)
                & (r < 0.35)
            ),
            (
                (r >= 0.35)
                & (r < 0.50)
            ),
            (
                (r >= 0.50)
                & (r <= 2.50)
            ),
        ],
        [
            0.00567,
            0.0100,
            0.0125,
            0.0172,
        ],
        default=np.nan,
    )


kappa_grid = (
    kappa_vent
    + kappa_dep(
        r_grid
    )
)


def construct_radius_distribution(
    r_median,
    GSD,
):
    emission_density = f_emit(
        r_grid,
        r_median,
        GSD,
    )

    exposure_weight = (
        emission_density
        / kappa_grid
        * (
            tau
            - (
                1.0
                - np.exp(
                    -kappa_grid
                    * tau
                )
            )
            / kappa_grid
        )
    )

    normalising_constant = np.trapezoid(
        exposure_weight,
        r_grid,
    )

    f_R_grid = (
        exposure_weight
        / normalising_constant
    )

    dr = (
        r_grid[1]
        - r_grid[0]
    )

    quad_weights = (
        f_R_grid
        * dr
    )

    quad_weights[0] *= 0.5
    quad_weights[-1] *= 0.5

    quad_weights /= (
        quad_weights.sum()
    )

    return (
        f_R_grid,
        quad_weights,
    )


(
    f_R_cal_grid,
    quad_weights_cal,
) = construct_radius_distribution(
    r_median_cal,
    GSD_cal,
)

(
    f_R_vivo_grid,
    quad_weights_vivo,
) = construct_radius_distribution(
    r_median_vivo,
    GSD_vivo,
)

# The radius-dependent delivery factor is shared.
radius_factor = (
    (4.0 / 3.0)
    * np.pi
    * r_grid**3
    * 1.0e-12
    * delta
)

print(
    "Robustness condition:"
)

print(
    "f_R_cal(r) != f_R_vivo(r)"
)

print(
    f"Calibration median radius = "
    f"{r_median_cal:.3f} micrometres"
)

print(
    f"In vivo median radius = "
    f"{r_median_vivo:.3f} micrometres"
)

print(
    f"Calibration quadrature weight sum = "
    f"{quad_weights_cal.sum():.12f}"
)

print(
    f"In vivo quadrature weight sum = "
    f"{quad_weights_vivo.sum():.12f}"
)

if not np.isclose(
    quad_weights_cal.sum(),
    1.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "Calibration quadrature weights do not sum to 1."
    )

if not np.isclose(
    quad_weights_vivo.sum(),
    1.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "In vivo quadrature weights do not sum to 1."
    )

In [ ]:
# @title Cell 5 — Define the exact marginal single-hit probability for each setting

def a_phi(
    phi,
    quad_weights,
):
    phi = float(
        phi
    )

    return float(
        np.dot(
            quad_weights,
            np.exp(
                -phi
                * radius_factor
            ),
        )
    )


def p_z1(
    phi,
    mu,
    quad_weights,
):
    mu = np.asarray(
        mu,
        dtype=float,
    )

    a = a_phi(
        phi,
        quad_weights,
    )

    p = (
        1.0
        - np.exp(
            mu
            * (
                a
                - 1.0
            )
        )
    )

    return np.clip(
        p,
        1.0e-12,
        1.0 - 1.0e-12,
    )


# Calibration probability uses f_R_cal(r).
p_cal_true = float(
    p_z1(
        phi_cal_true,
        mu_cal,
        quad_weights_cal,
    )
)

# In vivo probabilities use f_R_vivo(r).
p_vivo_true = np.asarray(
    p_z1(
        phi_vivo_true,
        mu_k,
        quad_weights_vivo,
    ),
    dtype=float,
)

print(
    f"a_cal(phi_cal_true) = "
    f"{a_phi(phi_cal_true, quad_weights_cal):.12f}"
)

print(
    f"P(Z=1; phi_cal_true, mu_cal) = "
    f"{p_cal_true:.6f}"
)

vivo_probability_check = pd.DataFrame(
    {
        "mu_k": mu_k,
        "P_Z1_true": p_vivo_true,
    }
)

display(
    vivo_probability_check
)

In [ ]:
# @title Cell 6 — Generate the simulation calibration outcome

rng_calibration = np.random.default_rng(
    SEED_CALIBRATION
)

Y_cal = int(
    rng_calibration.binomial(
        n=K_cal,
        p=p_cal_true,
    )
)

calibration_grouped = pd.DataFrame(
    {
        "mu": [
            mu_cal
        ],
        "N_group": [
            K_cal
        ],
        "Y_group": [
            Y_cal
        ],
        "true_probability": [
            p_cal_true
        ],
        "observed_probability": [
            Y_cal
            / K_cal
        ],
    }
)

display(
    calibration_grouped
)

print(
    f"Calibration outcome: "
    f"Y_cal = {Y_cal} "
    f"of K_cal = {K_cal}"
)

In [ ]:
# @title Cell 7 — Generate the simulation in vivo outcomes

rng_vivo = np.random.default_rng(
    SEED_VIVO
)

Y_k = rng_vivo.binomial(
    n=N_k,
    p=p_vivo_true,
)

in_vivo_grouped = pd.DataFrame(
    {
        "mu": mu_k,
        "N_group": N_k,
        "Y_group": Y_k.astype(
            int
        ),
        "true_probability": p_vivo_true,
        "observed_probability": (
            Y_k
            / N_k
        ),
    }
)

display(
    in_vivo_grouped
)

print(
    f"Total in vivo replicate trials = "
    f"{int(N_k.sum())}"
)

In [ ]:
# @title Cell 8 — Write the current one-parameter Stan model

stan_code = r"""
data {
  int<lower=1> K;
  array[K] int<lower=1> N_group;
  array[K] int<lower=0> Y_group;
  vector[K] mu_group;

  int<lower=2> N_R;
  vector[N_R] quad_weight;
  vector[N_R] radius_factor;

  real<lower=0> phi_reference;
}

parameters {
  real eta_phi;
}

transformed parameters {
  real<lower=0> phi;

  phi =
    phi_reference
    * exp(eta_phi);
}

model {
  real a_phi;

  // Broad prior on log(phi / phi_reference).
  eta_phi ~ normal(0, 1.5);

  a_phi =
    dot_product(
      quad_weight,
      exp(
        -phi
        * radius_factor
      )
    );

  for (k in 1:K) {
    real p_z1;

    p_z1 =
      1.0
      - exp(
          mu_group[k]
          * (a_phi - 1.0)
        );

    p_z1 =
      fmin(
        1.0 - 1e-12,
        fmax(
          1e-12,
          p_z1
        )
      );

    Y_group[k]
      ~ binomial(
          N_group[k],
          p_z1
        );
  }
}
"""

stan_file = (
    OUTPUT_DIR
    / "single_hit_phi_model.stan"
)

stan_file.write_text(
    stan_code,
    encoding="utf-8",
)

print(
    f"Stan model written to:\n"
    f"{stan_file}"
)

In [ ]:
# @title Cell 9 — Compile the Stan model and define shared helper functions

model = CmdStanModel(
    stan_file=str(
        stan_file
    )
)


def make_stan_data(
    grouped_df,
    quad_weights_input,
):
    return {
        "K": int(
            len(
                grouped_df
            )
        ),
        "N_group": (
            grouped_df[
                "N_group"
            ]
            .astype(
                int
            )
            .tolist()
        ),
        "Y_group": (
            grouped_df[
                "Y_group"
            ]
            .astype(
                int
            )
            .tolist()
        ),
        "mu_group": (
            grouped_df[
                "mu"
            ]
            .astype(
                float
            )
            .tolist()
        ),
        "N_R": int(
            N_R
        ),
        "quad_weight": (
            np.asarray(
                quad_weights_input,
                dtype=float,
            )
            .tolist()
        ),
        "radius_factor": (
            radius_factor.tolist()
        ),
        "phi_reference": float(
            phi_reference
        ),
    }


def summarize_phi_fit(
    fit,
    phi_true,
    label,
    max_treedepth=10,
):
    method = (
        fit
        .method_variables()
    )

    divergent = np.asarray(
        method[
            "divergent__"
        ]
    )

    treedepth = np.asarray(
        method[
            "treedepth__"
        ]
    )

    phi_draws = (
        fit
        .draws_pd(
            vars=[
                "phi"
            ]
        )[
            "phi"
        ]
        .to_numpy(
            dtype=float
        )
    )

    idata = az.from_cmdstanpy(
        posterior=fit
    )

    diagnostic_summary = az.summary(
        idata,
        var_names=[
            "eta_phi",
            "phi",
        ],
        hdi_prob=0.95,
    )

    q025 = float(
        np.quantile(
            phi_draws,
            0.025,
        )
    )

    q975 = float(
        np.quantile(
            phi_draws,
            0.975,
        )
    )

    recovery_summary = pd.DataFrame(
        [
            {
                "analysis": label,
                "phi_true": float(
                    phi_true
                ),
                "phi_posterior_mean": float(
                    phi_draws.mean()
                ),
                "phi_posterior_median": float(
                    np.median(
                        phi_draws
                    )
                ),
                "phi_q025": q025,
                "phi_q975": q975,
                "true_inside_95": bool(
                    q025
                    <= phi_true
                    <= q975
                ),
                "divergences": int(
                    divergent.sum()
                ),
                "max_treedepth_hits": int(
                    np.sum(
                        treedepth
                        >= max_treedepth
                    )
                ),
                "max_treedepth_fraction": float(
                    np.mean(
                        treedepth
                        >= max_treedepth
                    )
                ),
            }
        ]
    )

    return (
        phi_draws,
        diagnostic_summary,
        recovery_summary,
    )


print(
    "Stan model compiled and helper functions defined."
)

In [ ]:
# @title Cell 10 — Fit the calibration data and recover the posterior for phi_cal

calibration_stan_data = make_stan_data(
    calibration_grouped,
    quad_weights_cal,
)

if CALIBRATION_CMDSTAN_DIR.exists():
    shutil.rmtree(
        CALIBRATION_CMDSTAN_DIR
    )

CALIBRATION_CMDSTAN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

start = time.time()

calibration_fit = model.sample(
    data=calibration_stan_data,
    chains=3,
    parallel_chains=3,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=SEED_HMC_CALIBRATION,
    output_dir=str(
        CALIBRATION_CMDSTAN_DIR
    ),
    show_progress=True,
    refresh=50,
    adapt_delta=0.95,
    max_treedepth=10,
    save_warmup=False,
)

elapsed = (
    time.time()
    - start
)

print(
    f"Calibration HMC elapsed time: "
    f"{elapsed / 60.0:.2f} min"
)

(
    phi_cal_draws,
    calibration_diagnostics,
    calibration_phi_summary,
) = summarize_phi_fit(
    fit=calibration_fit,
    phi_true=phi_cal_true,
    label="calibration",
)

display(
    calibration_diagnostics
)

display(
    calibration_phi_summary
)

In [ ]:
# @title Cell 11 — Transform the calibration posterior from phi_cal to q

q_draws = (
    phi_cal_draws
    / n_cal_true
)

q_q025 = float(
    np.quantile(
        q_draws,
        0.025,
    )
)

q_q975 = float(
    np.quantile(
        q_draws,
        0.975,
    )
)

q_summary = pd.DataFrame(
    [
        {
            "parameter": "q",
            "true_value": float(
                q_true
            ),
            "posterior_mean": float(
                q_draws.mean()
            ),
            "posterior_median": float(
                np.median(
                    q_draws
                )
            ),
            "q025": q_q025,
            "q975": q_q975,
            "true_inside_95": bool(
                q_q025
                <= q_true
                <= q_q975
            ),
            "absolute_error": float(
                abs(
                    np.median(
                        q_draws
                    )
                    - q_true
                )
            ),
            "relative_error_percent": float(
                (
                    np.median(
                        q_draws
                    )
                    - q_true
                )
                / q_true
                * 100.0
            ),
        }
    ]
)

display(
    q_summary
)

print(
    "q is obtained from the calibration posterior as "
    "phi_cal / n_cal_true."
)

In [ ]:
# @title Cell 12 — Fit the in vivo data and recover the posterior for phi_vivo

in_vivo_stan_data = make_stan_data(
    in_vivo_grouped,
    quad_weights_vivo,
)

if VIVO_CMDSTAN_DIR.exists():
    shutil.rmtree(
        VIVO_CMDSTAN_DIR
    )

VIVO_CMDSTAN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

start = time.time()

in_vivo_fit = model.sample(
    data=in_vivo_stan_data,
    chains=3,
    parallel_chains=3,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=SEED_HMC_VIVO,
    output_dir=str(
        VIVO_CMDSTAN_DIR
    ),
    show_progress=True,
    refresh=50,
    adapt_delta=0.95,
    max_treedepth=10,
    save_warmup=False,
)

elapsed = (
    time.time()
    - start
)

print(
    f"In vivo HMC elapsed time: "
    f"{elapsed / 60.0:.2f} min"
)

(
    phi_vivo_draws,
    in_vivo_diagnostics,
    in_vivo_phi_summary,
) = summarize_phi_fit(
    fit=in_vivo_fit,
    phi_true=phi_vivo_true,
    label="in_vivo",
)

display(
    in_vivo_diagnostics
)

display(
    in_vivo_phi_summary
)

In [ ]:
# @title Cell 13 — Combine independent posterior draws and recover n_vivo

# The calibration and in vivo datasets are independent.
# Draw independently from the two posterior samples to
# propagate uncertainty from both phi_cal and phi_vivo.

N_COMBINED_DRAWS = 20000

rng_combination = np.random.default_rng(
    SEED_POSTERIOR_COMBINATION
)

calibration_indices = rng_combination.integers(
    low=0,
    high=len(
        q_draws
    ),
    size=N_COMBINED_DRAWS,
)

in_vivo_indices = rng_combination.integers(
    low=0,
    high=len(
        phi_vivo_draws
    ),
    size=N_COMBINED_DRAWS,
)

q_combined_draws = (
    q_draws[
        calibration_indices
    ]
)

phi_vivo_combined_draws = (
    phi_vivo_draws[
        in_vivo_indices
    ]
)

n_vivo_draws = (
    phi_vivo_combined_draws
    / q_combined_draws
)

n_vivo_q025 = float(
    np.quantile(
        n_vivo_draws,
        0.025,
    )
)

n_vivo_q975 = float(
    np.quantile(
        n_vivo_draws,
        0.975,
    )
)

n_vivo_summary = pd.DataFrame(
    [
        {
            "parameter": "n_vivo",
            "true_value": float(
                n_vivo_true
            ),
            "posterior_mean": float(
                n_vivo_draws.mean()
            ),
            "posterior_median": float(
                np.median(
                    n_vivo_draws
                )
            ),
            "q025": n_vivo_q025,
            "q975": n_vivo_q975,
            "true_inside_95": bool(
                n_vivo_q025
                <= n_vivo_true
                <= n_vivo_q975
            ),
            "absolute_error": float(
                abs(
                    np.median(
                        n_vivo_draws
                    )
                    - n_vivo_true
                )
            ),
            "relative_error_percent": float(
                (
                    np.median(
                        n_vivo_draws
                    )
                    - n_vivo_true
                )
                / n_vivo_true
                * 100.0
            ),
        }
    ]
)

display(
    n_vivo_summary
)

In [ ]:
# @title Cell 14 — Assemble the primary robustness analysis summary

primary_recovery_summary = pd.concat(
    [
        q_summary,
        n_vivo_summary,
    ],
    ignore_index=True,
)

display(
    primary_recovery_summary
)

print(
    "Robustness condition:"
)

print(
    "f_R_cal(r) != f_R_vivo(r)"
)

print(
    "\nPrimary recovery targets:"
)

print(
    "1. Recover q from calibration using f_R_cal(r)."
)

print(
    "2. Recover n_vivo using the calibrated q "
    "and f_R_vivo(r)."
)

## Expected manuscript results

With the fixed seeds and settings used above, this analysis should reproduce the manuscript's different-radius-distribution results:

- calibration emitted median radius: \(0.180\,\mu\mathrm{m}\);
- in vivo emitted median radius: \(0.139\,\mu\mathrm{m}\);
- GSD: 1.64 in both settings;
- posterior median \(q \approx 0.06662\);
- 95% posterior interval for \(q\): approximately 0.05530–0.07891;
- posterior median \(n_{\mathrm{vivo}} \approx 5.300\times10^{11}\) virions/mL;
- 95% posterior interval for \(n_{\mathrm{vivo}}\): approximately \(4.308\times10^{11}\) to \(6.586\times10^{11}\) virions/mL.

Generated Stan CSV files are written to the local `outputs/` directory and do not need to be committed to the repository.